# 02 — KPI uplift: baseline vs Aura

**Purpose.** Test whether the prototype reduces effort vs the participant's existing toolkit.
**KPI validated.** Effort reduction target ≥30% (`plan.md` section 22).

Method: paired comparison on `duration_sec` and `tap_count`. Shapiro-Wilk on the differences picks paired t-test (normal) vs Wilcoxon signed-rank (skewed). Cohen's d reported alongside the test.


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 200
ACCENT = "#FF5B2E"

SEED = 20260507
rng = np.random.default_rng(SEED)
CHART_DIR = Path("../charts"); CHART_DIR.mkdir(exist_ok=True)
DERIVED = Path("../../derived")


## Data load

Reuses `all_tasks.csv` if present, else synthesises a paired baseline-vs-prototype panel.


## SYNTHETIC DATA — REPLACE WITH REAL
Same generator as notebook 01. Seed = 20260507. Replace the synth call with the real CSV load.


In [ ]:
def synth_tasks(n=30):
    ids = [f"P{i:03d}" for i in range(1, n + 1)]
    rows = []
    for pid in ids:
        for task_id in range(1, 6):
            base_mu_t = [180, 95, 140, 240, 55][task_id - 1]
            proto_mu_t = [110, 55, 80, 175, 28][task_id - 1]
            base_mu_tap = [42, 18, 24, 8, 14][task_id - 1]
            proto_mu_tap = [22, 9, 11, 4, 6][task_id - 1]
            rows.append({"participant_id": pid, "round": "baseline", "task_id": task_id,
                         "duration_sec": max(5.0, rng.normal(base_mu_t, base_mu_t * 0.18)),
                         "tap_count": max(1, int(rng.normal(base_mu_tap, max(2.0, base_mu_tap * 0.20))))})
            rows.append({"participant_id": pid, "round": "prototype", "task_id": task_id,
                         "duration_sec": max(5.0, rng.normal(proto_mu_t, proto_mu_t * 0.18)),
                         "tap_count": max(1, int(rng.normal(proto_mu_tap, max(2.0, proto_mu_tap * 0.20))))})
    return pd.DataFrame(rows)

if (DERIVED / "all_tasks.csv").exists():
    tasks = pd.read_csv(DERIVED / "all_tasks.csv")
else:
    tasks = synth_tasks()
print("rows:", len(tasks))


## Build paired panel

Pivot to wide on `round` so each row holds one participant-task with a baseline value and a prototype value.


In [ ]:
def pivot_metric(df, metric):
    return (
        df.pivot_table(index=["participant_id", "task_id"], columns="round", values=metric)
          .dropna()
          .reset_index()
    )

dur = pivot_metric(tasks, "duration_sec")
tap = pivot_metric(tasks, "tap_count")
dur["delta_pct"] = (dur["baseline"] - dur["prototype"]) / dur["baseline"] * 100
tap["delta_pct"] = (tap["baseline"] - tap["prototype"]) / tap["baseline"] * 100
print("paired duration rows:", len(dur), "| paired tap rows:", len(tap))
dur.head()


## Statistical tests

Per task, run Shapiro-Wilk on the differences. If p > 0.05 use paired t-test, else Wilcoxon. Report Cohen's d and a 95% CI on the mean difference.


In [ ]:
def cohens_d_paired(diffs):
    d = np.asarray(diffs, dtype=float)
    sd = d.std(ddof=1)
    return float(d.mean() / sd) if sd > 0 else 0.0

def paired_test(panel, metric_label):
    results = []
    for tid, sub in panel.groupby("task_id"):
        diffs = sub["baseline"].values - sub["prototype"].values
        n = len(diffs)
        shap_p = stats.shapiro(diffs).pvalue if n >= 3 else 1.0
        if shap_p > 0.05:
            test_name = "paired_t"
            stat, p = stats.ttest_rel(sub["baseline"], sub["prototype"])
        else:
            test_name = "wilcoxon"
            stat, p = stats.wilcoxon(sub["baseline"], sub["prototype"])
        m = float(np.mean(diffs))
        se = float(np.std(diffs, ddof=1) / np.sqrt(n))
        lo, hi = m - 1.96 * se, m + 1.96 * se
        results.append({
            "task_id": tid,
            "metric": metric_label,
            "n": n,
            "mean_delta": round(m, 3),
            "ci95_lo": round(lo, 3),
            "ci95_hi": round(hi, 3),
            "test": test_name,
            "stat": round(float(stat), 3),
            "p_value": float(f"{p:.4g}"),
            "cohens_d": round(cohens_d_paired(diffs), 3),
            "mean_pct_reduction": round(sub["delta_pct"].mean(), 2),
        })
    return pd.DataFrame(results)

dur_results = paired_test(dur, "duration_sec")
tap_results = paired_test(tap, "tap_count")
print("--- duration ---"); print(dur_results.to_string(index=False))
print("\n--- taps ---"); print(tap_results.to_string(index=False))


## Headline KPI: ≥30% effort reduction

Pool across tasks: mean percent reduction in duration. Report whether the lower bound of the 95% CI clears 30%.


In [ ]:
def headline(panel, label):
    pct = panel["delta_pct"].values
    n = len(pct)
    m = pct.mean()
    se = pct.std(ddof=1) / np.sqrt(n)
    lo, hi = m - 1.96 * se, m + 1.96 * se
    print(f"{label}: mean reduction = {m:.1f}% (95% CI [{lo:.1f}%, {hi:.1f}%]) over n={n} pairs")
    print(f"  meets >=30% target: {lo >= 30}")
    return m, lo, hi

dur_head = headline(dur, "Duration")
tap_head = headline(tap, "Tap count")


## Charts

Per-task mean difference with 95% CI bars; one for duration, one for taps.


In [ ]:
def forest(results, title, fname, unit):
    fig, ax = plt.subplots(figsize=(7, 4.2))
    y = np.arange(len(results))
    ax.errorbar(
        results["mean_delta"], y,
        xerr=[results["mean_delta"] - results["ci95_lo"], results["ci95_hi"] - results["mean_delta"]],
        fmt="o", color=ACCENT, capsize=4, linewidth=1.8,
    )
    ax.axvline(0, color="#0E0E0E", linewidth=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels([f"Task {t}" for t in results["task_id"]])
    ax.set_xlabel(f"Baseline minus prototype ({unit})")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(CHART_DIR / fname)
    plt.show()

forest(dur_results, "Duration reduction by task (mean and 95% CI)", "02_duration_forest.png", "sec")
forest(tap_results, "Tap-count reduction by task (mean and 95% CI)", "02_taps_forest.png", "taps")


## Results — interpretation

- On synthetic data, every task shows a positive baseline-minus-prototype delta with a 95% CI excluding zero.
- Headline percent reduction across all tasks beats the 30% target on synthetic; real-pilot reading goes here.
- Cohen's d magnitudes flag whether the win is "barely detectable" (d < 0.2) or "obvious" (d > 0.8). Synthetic tuning lands in the 0.8-2.0 range across tasks; real data may differ.
